# FSBL example (host + JAX d1 static limb)

Finite-source binary-lens photometry from `bagle.model_jax`, compared to the matching point-source (`PSBL`) model and the milestone (d1) static-limb kernel in `bagle.jax.fsbl`.

**Units (phot+astrom)**
- `sep` in mas; `radiusS` in arcsec; convert with `rho = radiusS * 1e3 / thetaE_amp`.

Milestone (d1): static `n_outline` limb (no adaptive insert / `jax.random`). Milestone (d2): microlux-style adaptive contour (stubbed).


In [ ]:
%matplotlib inline
from bagle import model_jax as model
from bagle.jax import fsbl as fsbl_jax
import numpy as np
import matplotlib.pyplot as plt


## Phot+astrom parameters (non-caustic)

Wide impact parameter so the trajectory misses the caustics. Finite-source effect is still visible versus PSBL.

| Param | Value | Units |
|---|---|---|
| $m_{Lp}, m_{Ls}$ | 0.5, 0.5 | $M_\odot$ |
| $t_0$ | 57000 | MJD |
| $\beta$ | 2.0 | mas |
| $d_L, d_S$ | 4000, 8000 | pc |
| $sep$ | 1.0 | mas |
| $\alpha$ | 45 | deg |
| $radiusS$ | $2\times10^{-4}$ | arcsec |
| $n_{outline}$ | 120 | |


In [ ]:
mLp, mLs = 0.5, 0.5
t0 = 57000.0
beta = 2.0  # mas; wide non-caustic
dL, dS = 4000.0, 8000.0
xS0_E, xS0_N = [0.0], [0.0]
muL_E, muL_N = 0.0, 0.0
muS_E, muS_N = 5.0, 0.0
radiusS = 2e-4  # arcsec
sep, alpha = 1.0, 45.0
b_sff, mag_src, dmag = [1.0], [18.0], [20.0]
n_outline = 120

fsbl = model.FSBL_PhotAstrom_noPar_Param1(
    mLp, mLs, t0, radiusS, xS0_E, xS0_N, beta,
    muL_E, muL_N, muS_E, muS_N, dL, dS, sep, alpha,
    b_sff, mag_src, dmag, n_outline=n_outline)
psbl = model.PSBL_PhotAstrom_noPar_Param1(
    mLp, mLs, t0, xS0_E, xS0_N, beta,
    muL_E, muL_N, muS_E, muS_N, dL, dS, sep, alpha,
    b_sff, mag_src, dmag)

rho = fsbl.radiusS * 1e3 / fsbl.thetaE_amp
print(f"thetaE={fsbl.thetaE_amp:.3f} mas, u0={fsbl.u0_amp:.3f}, rho={rho:.3f}, tE={fsbl.tE:.1f} d")
t = np.linspace(t0 - 1.5 * fsbl.tE, t0 + 1.5 * fsbl.tE, 200)


### 1. Light curve (host FSBL vs PSBL vs JAX d1)


In [ ]:
amp_f = np.asarray(fsbl.get_amplification(t))
amp_p = np.asarray(psbl.get_amplification(t))

# JAX d1 static limb on the same geometry
u_vec = np.asarray(fsbl.get_u(t))
rho = float(fsbl.source_radius_thetaE_units())
lens_res = np.asarray(fsbl.get_resolved_lens_astrometry(t))
lens1 = lens_res[0] / fsbl.thetaE_amp * 1e3
lens2 = lens_res[1] / fsbl.thetaE_amp * 1e3
w = u_vec[:, 0] + 1j * u_vec[:, 1]
z1_abs = lens1[:, 0] + 1j * lens1[:, 1]
z2_abs = lens2[:, 0] + 1j * lens2[:, 1]
z_cm = 0.5 * (z1_abs + z2_abs)
m1 = fsbl.mLp / fsbl.mL
m2 = fsbl.mLs / fsbl.mL
amp_j = np.asarray(fsbl_jax.fsbl_magnification(
    w, z1_abs - z_cm, z2_abs - z_cm, m1, m2, rho, n_outline))

fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(t, amp_f, label="FSBL host")
axes[0].plot(t, amp_p, ls=":", label="PSBL")
axes[0].plot(t, amp_j, ls="--", label="FSBL JAX d1")
axes[0].set_ylabel("Amplification")
axes[0].legend()
axes[1].plot(t, amp_j - amp_f, label="JAX d1 - host")
axes[1].plot(t, amp_f - amp_p, label="FSBL - PSBL")
axes[1].set_xlabel("MJD")
axes[1].set_ylabel("Difference")
axes[1].legend()
plt.show()
print(f"max |d1 - host| / host = {np.max(np.abs(amp_j - amp_f) / amp_f):.3e}")
